In [7]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import datetime

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

from ai_agents.agents.email_finder import run_batch, run_single, SourceType, LeadStatus
from ai_agents.agents.email_finder.graph import build_graph
from google_utils.google_sheet import GoogleSheetService

from dotenv import load_dotenv
load_dotenv()


CHECKPOINT_FILE = "email_finder_checkpoint.json"
BATCH_SIZE = 50
CONCURRENCY = 5

In [2]:
# ── v2 Graph structure ──
# canonical_builder
#   ├─ (has existing_email, no website) → validate_existing_email
#   │     ├─ (email validated ≥0.85) → END
#   │     ├─ (has website) → discover_urls
#   │     └─ (no website) → perplexity_discovery
#   ├─ (has website) → discover_urls → crawl_page (fan-out) → resolve_best_email
#   │     ├─ (email found) → END
#   │     ├─ (no candidates + has FB link) → fb_crawler
#   │     │     ├─ (email found) → END
#   │     │     └─ → perplexity_discovery → END
#   │     └─ → perplexity_discovery → END
#   └─ (no website) → perplexity_discovery → END

graph = build_graph()
print("Graph compiled OK")
print(f"Nodes: {list(graph.get_graph().nodes)}")

# Render the graph as ASCII (or use graph.get_graph().draw_mermaid() for Mermaid)
try:
    print(graph.get_graph().draw_ascii())
except Exception:
    print(graph.get_graph().draw_mermaid())

Graph compiled OK
Nodes: ['__start__', 'canonical_builder', 'validate_existing_email', 'discover_urls', 'crawl_page', 'resolve_best_email', 'perplexity_discovery', 'fb_crawler', '__end__']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	canonical_builder(canonical_builder)
	validate_existing_email(validate_existing_email)
	discover_urls(discover_urls)
	crawl_page(crawl_page)
	resolve_best_email(resolve_best_email)
	perplexity_discovery(perplexity_discovery)
	fb_crawler(fb_crawler)
	__end__([<p>__end__</p>]):::last
	__start__ --> canonical_builder;
	canonical_builder -.-> discover_urls;
	canonical_builder -.-> perplexity_discovery;
	canonical_builder -.-> validate_existing_email;
	crawl_page --> resolve_best_email;
	discover_urls -.-> crawl_page;
	discover_urls -.-> resolve_best_email;
	fb_crawler -.-> __end__;
	fb_crawler -.-> perplexity_discovery;
	resolve_best_email -.-> __end__;
	resolve_best_email -.-> fb_crawler;
	resolve_best_email

In [3]:
def _format_email_source(best_email: dict) -> str:
    source = (best_email.get("source") or "").strip()
    note = (best_email.get("note") or "").strip()
    if source and note:
        return f"{source} — {note}"
    if note:
        return note
    return source


def result_to_sheet_row(raw_row: dict, result: dict) -> list:
    best_email = result.get("best_email") or {}
    if isinstance(best_email, dict):
        email = best_email.get("email", "")
        source_display = _format_email_source(best_email)
        confidence = best_email.get("confidence", "")
    else:
        email = ""
        source_display = ""
        confidence = ""

    nodes = result.get("nodes_executed") or []
    errors = result.get("errors") or []

    return [
        raw_row.get("Podcast Name", ""),
        raw_row.get("Podcast Website", ""),
        email,
        source_display,
        str(confidence),
        result.get("status", ""),
        " → ".join(nodes),
        "; ".join(errors) if errors else "",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ]

In [4]:
def _create_sheet_tab(sheet_service: GoogleSheetService, spreadsheet_id: str, sheet_name: str) -> None:
    """Create a new sheet tab if it doesn't exist."""
    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if success and sheet_name in sheet_names:
        return
    sheet_service.service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},
    ).execute()
    print(f"Created sheet tab '{sheet_name}'")


def _ensure_email_finder_headers(
    sheet_service: GoogleSheetService,
    spreadsheet_id: str,
    sheet_name: str,
) -> None:
    """Create the Email_Finder sheet tab (if needed) and add headers if empty."""
    _create_sheet_tab(sheet_service, spreadsheet_id, sheet_name)
    success, values = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A1:I1")
    if not success or not values:
        headers = [[
            "Podcast Name",
            "Podcast Website",
            "Email Found",
            "Email Source",
            "Confidence",
            "Status",
            "Nodes Executed",
            "Errors",
            "Processed At",
        ]]
        sheet_service.append_rows(spreadsheet_id, sheet_name, headers)
        print(f"Added headers to {sheet_name}")


def run_email_finder_pipeline(
    spreadsheet_id: str,
    source_type: SourceType,
    needs_enrichment_sheet: str = "Needs_enrichment",
    email_finder_sheet: str = "Email_Finder",
    batch_size: int = BATCH_SIZE,
    concurrency: int = CONCURRENCY,
    dry_run: bool = False,
) -> dict:
    """
    Main pipeline entry point (v2).

    v2 graph flow:
      canonical_builder → validate_existing_email (if existing email, no website)
                        → discover_urls → crawl_page (fan-out) → resolve_best_email
                        → fb_crawler (if no candidates but FB link found)
                        → perplexity_discovery (fallback)

    Args:
        spreadsheet_id: Google Sheet ID
        source_type: SourceType enum value
        needs_enrichment_sheet: Sheet to read from
        email_finder_sheet: Sheet to write found emails to
        batch_size: Rows per batch
        concurrency: Concurrent leads per batch
        dry_run: If True, don't write to sheets (for testing)
    """
    sheet_service = GoogleSheetService()
    checkpoint = load_checkpoint(spreadsheet_id)
    already_processed = set(checkpoint["processed_indices"])

    # ── Read all rows ──
    print(f"Reading {needs_enrichment_sheet}...")
    success, df = sheet_service.get_sheet_data(spreadsheet_id, needs_enrichment_sheet)
    assert success, f"Failed to read sheet: {df}"

    total_rows = len(df)
    print(f"Total rows: {total_rows} | Already processed: {len(already_processed)}")

    # Filter out already processed rows
    remaining_indices = [i for i in range(total_rows) if i not in already_processed]
    print(f"Remaining to process: {len(remaining_indices)}")

    if not remaining_indices:
        print("Nothing to process — all rows already handled")
        return checkpoint

    # Ensure Email_Finder sheet has headers if empty
    if not dry_run:
        _ensure_email_finder_headers(sheet_service, spreadsheet_id, email_finder_sheet)

    # ── Per-node stats (v2) ──
    node_stats: Counter = Counter()
    source_stats: Counter = Counter()

    # ── Process in batches ──
    all_processed_indices = list(checkpoint["processed_indices"])

    for batch_start in range(0, len(remaining_indices), batch_size):
        batch_indices = remaining_indices[batch_start: batch_start + batch_size]
        batch_rows = [df.iloc[i].to_dict() for i in batch_indices]

        print(f"\nBatch {batch_start // batch_size + 1} — rows {batch_indices[0]+1} to {batch_indices[-1]+1}")

        # Run batch through email finder
        results = run_batch(
            rows=batch_rows,
            source_type=source_type,
            concurrency=concurrency,
        )

        # Separate found vs not found — only commit found indices after successful sheet write
        found_rows = []
        found_indices = []

        for row_idx, raw_row, result in zip(batch_indices, batch_rows, results):
            status = result.get("status", "")

            # Track which nodes executed (v2)
            for node in (result.get("nodes_executed") or []):
                node_stats[node] += 1

            # Track email source
            best = result.get("best_email") or {}
            if isinstance(best, dict) and best.get("source"):
                source_stats[best["source"].split(" — ")[0]] += 1

            if status == "email_found":
                found_rows.append(result_to_sheet_row(raw_row, result))
                found_indices.append(row_idx)
                checkpoint["found_count"] += 1
            else:
                # Non-email rows are safe to mark processed immediately
                all_processed_indices.append(row_idx)
                checkpoint["processed_indices"].append(row_idx)
                if status == "failed":
                    checkpoint["failed_count"] += 1
                else:
                    checkpoint["not_found_count"] += 1

        # Write found emails to Email_Finder sheet
        if found_rows:
            if not dry_run:
                success, msg = sheet_service.append_rows(
                    spreadsheet_id, email_finder_sheet, found_rows
                )
                print(f"  → Wrote {len(found_rows)} emails to {email_finder_sheet}: {msg}")
                if success:
                    all_processed_indices.extend(found_indices)
                    checkpoint["processed_indices"].extend(found_indices)
                else:
                    # Sheet write failed — don't mark as processed so they'll be retried
                    checkpoint["found_count"] -= len(found_rows)
                    print(f"  ⚠ Sheet write failed — {len(found_rows)} found emails will be retried next run")
            else:
                all_processed_indices.extend(found_indices)
                checkpoint["processed_indices"].extend(found_indices)

        # Save checkpoint after every batch
        save_checkpoint(checkpoint)
        print(f"  → Checkpoint saved | Found: {checkpoint['found_count']} | Not found: {checkpoint['not_found_count']} | Failed: {checkpoint['failed_count']}")

    # ── Remove processed rows from Needs_Enrichment ──
    if all_processed_indices and not dry_run:
        print(f"\nRemoving {len(all_processed_indices)} processed rows from {needs_enrichment_sheet}...")
        success, msg = sheet_service.delete_rows_by_indices(
            spreadsheet_id,
            needs_enrichment_sheet,
            all_processed_indices,
        )
        print(f"  → {msg}")

    # ── Final summary ──
    print(f"\n{'='*50}")
    print(f"Pipeline complete")
    print(f"{'='*50}")
    print(f"  Found:     {checkpoint['found_count']}")
    print(f"  Not found: {checkpoint['not_found_count']}")
    print(f"  Failed:    {checkpoint['failed_count']}")

    if node_stats:
        print(f"\n  Node execution counts:")
        for node, count in node_stats.most_common():
            print(f"    {node}: {count}")

    if source_stats:
        print(f"\n  Email sources:")
        for source, count in source_stats.most_common():
            print(f"    {source}: {count}")

    if not dry_run:
        clear_checkpoint()

    return checkpoint

In [5]:
# ── Quick single-lead test (v2) ──
# Use this to test the new graph paths before running the full pipeline.
# Uncomment the scenario you want to test.

# Scenario 1: Lead with existing email + no website → validate_existing_email path
# test_row = {"Host Name": "Test Host", "Email": "host@example.com"}

# Scenario 2: Lead with website → discover_urls → crawl_page → resolve path
# test_row = {"Host Name": "Test Host", "Podcast Website": "https://example.com"}

# Scenario 3: Lead with FB link → fb_crawler path (if scraping finds no emails)
# test_row = {"Host Name": "Test Host", "Podcast Website": "https://example.com", "Facebook": "https://facebook.com/example"}

# Scenario 4: No website, no email → perplexity_discovery fallback
# test_row = {"Host Name": "Test Host", "Podcast Name": "Example Podcast"}

# result = run_single(test_row, SourceType.PODSCAN_HOST)
# print(f"Status: {result.get('status')}")
# print(f"Nodes:  {' → '.join(result.get('nodes_executed', []))}")
# print(f"Email:  {result.get('best_email', {})}")
# print(f"Errors: {result.get('errors', [])}")

In [6]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1TL5FEYrzW6zzUxV2U-PDX5LRBWjfi3A9xGbK5bga-E0/edit?gid=1886524830#gid=1886524830"
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

result = run_email_finder_pipeline(
    spreadsheet_id=spreadsheet_id,
    source_type=SourceType.PODSCAN_HOST,
    dry_run=False,
    batch_size=50,
    concurrency=5,
)

NameError: name 'load_checkpoint' is not defined